# Distracted Driver Detection (PyTorch) - local Windows / GTX 1650 version

Expected folder layout next to this notebook:
```
data/imgs/train/c0 ... c9
data/imgs/test/*.jpg
```

### 1. Imports and settings

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# ---- Local paths (changed from the Kaggle paths) ----
TRAIN_PATH = os.path.join('data', 'imgs', 'train')
TEST_PATH = os.path.join('data', 'imgs', 'test')

# ---- Settings for a 4 GB GPU (GTX 1650) ----
BATCH_SIZE = 32      # lower to 16 if you get 'CUDA out of memory'
NUM_WORKERS = 2      # set to 0 if you get multiprocessing errors in Jupyter
IMG_SIZE = 150
NUM_EPOCHS = 6

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

### 2. Look at the data

In [ ]:
for folder in sorted(glob.glob(os.path.join(TRAIN_PATH, '*'))):
    print(f'{os.path.basename(folder)} : {len(os.listdir(folder))} images')

In [ ]:
classes = {
    'c0': 'safe driving',
    'c1': 'texting - right',
    'c2': 'talking on the phone - right',
    'c3': 'texting - left',
    'c4': 'talking on the phone - left',
    'c5': 'operating the radio',
    'c6': 'drinking',
    'c7': 'reaching behind',
    'c8': 'hair and makeup',
    'c9': 'talking to passenger'
}

def show_images(folder_path, num):
    images = glob.glob(os.path.join(folder_path, '*'))
    selected = np.random.choice(images, num, replace=False)
    fig = plt.figure(figsize=(10, 6))
    columns = 3
    rows = (len(selected) // columns) + 1
    for i, image_file in enumerate(selected):
        img = plt.imread(image_file)
        ax = fig.add_subplot(rows, columns, i + 1)
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle(classes[os.path.basename(folder_path)])
    plt.show()

# one class only, to keep it quick. Loop over all folders if you want all 10.
show_images(os.path.join(TRAIN_PATH, 'c1'), 6)

### 3. Transforms and data loaders

The original notebook assigned `.transform` on a dataset shared by train and validation, so both ended up with the same transform. Here we make two separate `ImageFolder` objects and split the **indices**, so training gets augmentation and validation does not.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(1.0, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_full = datasets.ImageFolder(root=TRAIN_PATH, transform=train_transform)
val_full = datasets.ImageFolder(root=TRAIN_PATH, transform=test_transform)

print(train_full.class_to_idx)

# 80/20 split on indices (fixed seed so results are repeatable)
g = torch.Generator().manual_seed(42)
perm = torch.randperm(len(train_full), generator=g).tolist()
n_train = int(0.8 * len(perm))
train_dataset = Subset(train_full, perm[:n_train])
val_dataset = Subset(val_full, perm[n_train:])
print('train:', len(train_dataset), 'val:', len(val_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'))

Note: horizontal flip was removed from the augmentation because it swaps left/right (e.g. 'texting - right' vs 'texting - left'), which would confuse the labels.

### 4. Training and plotting helpers

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device='cuda'):
    model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        # ---- training ----
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        avg_train_loss = train_loss / len(train_loader)

        # ---- validation ----
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        val_acc = 100 * correct / total
        avg_val_loss = val_loss / len(val_loader)

        print(f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)

    return model, history


def plot_performance(history):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    plt.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'b-', label='Training Accuracy')
    plt.plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs'); plt.ylabel('Accuracy (%)'); plt.legend()

    plt.tight_layout()
    plt.show()

### 5. Baseline CNN

The huge 480x640 dense baseline from the original notebook is left out because it does not fit in 4 GB of GPU memory.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)    # identity path
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 37 * 37, 512)   # 150 -> 75 -> 37 after two pools
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        identity = torch.relu(self.conv(x))
        out = torch.relu(self.conv1(x))
        out = torch.relu(self.conv2(out))
        out = out + identity
        out = self.pool(self.pool(out))
        out = out.view(out.size(0), -1)
        out = torch.relu(self.fc1(out))
        return self.fc2(out)

model = BaselineCNN(num_classes=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
trained_model, history = train_model(model, train_loader, val_loader, criterion, optimizer,
                                     num_epochs=NUM_EPOCHS, device=device)
torch.save(trained_model.state_dict(), 'baseline_cnn.pth')

In [ ]:
plot_performance(history)

### 6. Transfer learning with ResNet18

In [ ]:
transfer_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
transfer_model.fc = nn.Linear(transfer_model.fc.in_features, 10)

# NEW optimizer for the new model (the original reused the CNN's optimizer by mistake)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(transfer_model.parameters(), lr=1e-4)

trained_transfer_model, history_tl = train_model(transfer_model, train_loader, val_loader, criterion,
                                                 optimizer, num_epochs=NUM_EPOCHS, device=device)
torch.save(trained_transfer_model.state_dict(), 'resnet18_driver.pth')

In [ ]:
plot_performance(history_tl)

### 7. Predict on the test folder

The test folder has ~79,000 images, so this is batched and can take a while. Use `LIMIT = 500` for a quick trial, or `None` for everything.

In [ ]:
LIMIT = 500   # set to None to predict all test images

def predict_folder(folder_path, model, limit=None, batch_size=64):
    model.eval()
    paths = sorted(glob.glob(os.path.join(folder_path, '*')))
    if limit:
        paths = paths[:limit]
    names, probs = [], []
    with torch.no_grad():
        for i in tqdm(range(0, len(paths), batch_size)):
            batch_paths = paths[i:i + batch_size]
            batch = torch.stack([test_transform(Image.open(p).convert('RGB')) for p in batch_paths]).to(device)
            p = torch.softmax(model(batch), dim=1).cpu().tolist()
            probs.extend(p)
            names.extend([os.path.basename(x) for x in batch_paths])
    return names, probs

images, probabilities = predict_folder(TEST_PATH, trained_transfer_model, limit=LIMIT)

final_sub = pd.concat((pd.DataFrame({'img': images}),
                       pd.DataFrame(probabilities, columns=[f'c{i}' for i in range(10)])), axis=1)
print(final_sub.isnull().sum().sum(), 'missing values')
final_sub.to_csv('final_sub.csv', index=False)
final_sub.head()